# Deep Q-learning against SARSA with function approximation

A port of the comparison in `lunardeepQSarsaSemiGradient.py` -- the code
behind the result plot on the deck's slide 25.

Unit 11 put Q-learning and SARSA side by side in the tabular case and found
they differ in exactly one symbol: the bootstrap. This notebook does the same
comparison once both have had their tables replaced by a neural network, on a
problem where a table is not an option at all.

$$\textbf{DQN}\quad z_k=g(x_k,u_k)+\alpha\max_{u}Q(x_{k+1},u;\bar\theta)$$

$$\textbf{SARSA}\quad z_k=g(x_k,u_k)+\alpha\,Q(x_{k+1},u_{k+1};\theta)$$

Three differences follow from that one line, and they are what this notebook
is about:

- DQN maximises over the next action, SARSA uses the action its
  $\epsilon$-greedy policy actually takes -- off-policy against on-policy;
- DQN can therefore reuse old transitions from a **replay buffer**, while
  SARSA cannot, because a stored transition carries the action an *older*
  policy chose;
- DQN needs a **target network** to keep the maximised target still; SARSA's
  update is a plain **semi-gradient** step with no second network.

::: Requirements :::
`torch` and `gymnasium`; Box2D for the lunar lander itself
(`pip install "gymnasium[box2d]"`), with an automatic fall-back to
`CartPole-v1`. The agent code is in `deep_rl.py` next to this notebook.

**Convention warning.** Gymnasium rewards are *maximised*; Units 1-11
minimise a *cost*. The greedy action here is an `argmax`.

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import deep_rl
from deep_rl import Config, seed_all, make_env, DQNAgent, SARSAAgent, \
                    train_dqn, train_sarsa, evaluate_agent, smooth

NUM_EPISODES = 800      # the deck's figure uses 800; 150-200 already shows the shape
cfg = Config(num_episodes=NUM_EPISODES)
print("device:", deep_rl.DEVICE)

## The two agents

Same network, same discount, same step size, same $\epsilon$ schedule. Only
the update differs -- and, as a consequence of the update, whether a replay
buffer and a target network are present at all.

In [ ]:
seed_all(cfg.seed)
env = make_env(cfg.env_id, seed=cfg.seed)
obs_dim, n_actions = env.observation_space.shape[0], env.action_space.n

dqn   = DQNAgent(obs_dim, n_actions, cfg)
sarsa = SARSAAgent(obs_dim, n_actions, cfg)

print(f"{env.spec.id}: {obs_dim} state dimensions, {n_actions} actions\n")
print(f"{'':<22}{'replay':>10}{'target net':>14}{'update':>22}")
print("-" * 70)
print(f"{'DQN':<22}{'yes':>10}{'yes':>14}{'max over next action':>22}")
print(f"{'SARSA':<22}{'no':>10}{'no':>14}{'action actually taken':>22}")

### Why SARSA has no replay buffer

Not an oversight, and not a simplification. A replayed transition
$(x,u,g,x')$ is enough for DQN, because its target maximises over the next
action and so does not care which policy generated the data. SARSA's target
needs $u'$ -- the action the policy took next -- and a stored $u'$ was chosen
by whatever policy was current when the transition was recorded, which is not
the policy being evaluated now.

This is exactly the observation Problem 11.4 makes about the data table being
one column short, restated as a property of the algorithm: Q-learning's
training example is a *set* member, SARSA's is part of a *sequence*.

In [ ]:
t0 = time.perf_counter()
dqn_returns, _ = train_dqn(dqn, env, cfg,
                           eval_env=make_env(cfg.env_id, seed=cfg.seed + 100, quiet=True),
                           verbose=True)
t_dqn = time.perf_counter() - t0
print(f"\nDQN: {len(dqn_returns)} episodes in {t_dqn:.0f} s")

In [ ]:
t0 = time.perf_counter()
sarsa_returns, _ = train_sarsa(sarsa, env, cfg,
                               eval_env=make_env(cfg.env_id, seed=cfg.seed + 200, quiet=True),
                               verbose=True)
t_sarsa = time.perf_counter() - t0
print(f"\nSARSA: {len(sarsa_returns)} episodes in {t_sarsa:.0f} s")

## The result

This is the figure on slide 25. Both raw curves are drawn faintly with their
moving averages on top; without the averages the two are impossible to tell
apart.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(dqn_returns,   color="C0", alpha=0.25, lw=1, label="DQN returns (train)")
ax.plot(sarsa_returns, color="C1", alpha=0.25, lw=1, label="SARSA returns (train)")
for r, c, lab in ((dqn_returns, "C0", "DQN, moving average"),
                  (sarsa_returns, "C1", "SARSA, moving average")):
    sm = smooth(r, 20)
    ax.plot(np.arange(len(sm)) + 19, sm, color=c, lw=2.2, label=lab)
ax.axhline(200, color="k", ls=":", lw=1)
ax.set_xlabel("Episode"); ax.set_ylabel("Episode return")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
print(f"{'':<34}{'last 100 mean':>15}{'best':>10}{'>= 200':>10}{'time (s)':>11}")
print("-" * 82)
for r, lab, t in ((dqn_returns, "DQN (off-policy, replay+target)", t_dqn),
                  (sarsa_returns, "SARSA (on-policy, semi-gradient)", t_sarsa)):
    last = np.array(r[-100:])
    print(f"{lab:<34}{last.mean():>15.1f}{max(r):>10.1f}"
          f"{np.mean(last >= 200):>9.0%}{t:>11.0f}")

## Reading the comparison

**They end up in much the same place.** That is the deck's claim, and it is
the same conclusion Unit 11 reached on the frozen lake. The reason is the one
given there: with an $\epsilon$-greedy behaviour policy and $\epsilon$ decaying
to $0.02$, the action SARSA bootstraps on *is* the maximising action around
$98\%$ of the time by the end of the run. The two targets agree whenever that
happens, so the two algorithms are only genuinely different on the $2\%$ of
steps where exploration intervenes.

**SARSA is usually the faster of the two in wall-clock time**, which is worth
naming because it looks backwards. SARSA does one gradient step on one
transition per environment step; DQN does one gradient step on a *minibatch of
$64$* per environment step. DQN is doing far more arithmetic per step of
experience -- that is what the replay buffer is for, and it is why DQN
generally needs fewer *episodes* for the same result even while needing more
compute per episode. Compare the number of episodes to a given score, not the
clock, if what you care about is sample efficiency.

**Neither has a convergence guarantee here.** Unit 10's theorem was tabular.
Once $Q$ is a network, the contraction argument is gone: SARSA's update is a
semi-gradient, which is not the gradient of any cost, and DQN's target moves
whenever the target network is refreshed. What the plot shows is that they
work on this problem, not that they must.

## The one place they genuinely differ

The averaged curves hide the distinction, because it only shows up where
exploration matters. A cheap way to make it visible is to raise the
exploration floor: with $\epsilon$ held at a larger value, SARSA -- which is
evaluating the policy it actually follows, exploration included -- should
prefer a more conservative policy than DQN, which evaluates the greedy policy
regardless of the random actions it is taking.

This is the lunar-lander version of Unit 11's frozen-lake observation that the
two settle on similar but not identical policies.

In [ ]:
CAUTIOUS_EPISODES = min(250, NUM_EPISODES)
HIGH_EPS = 0.20          # exploration floor held well above the usual 0.02

def run(kind, eps_end):
    c = Config(num_episodes=CAUTIOUS_EPISODES,
               dqn_eps_end=eps_end, sarsa_eps_end=eps_end)
    seed_all(c.seed)
    e = make_env(c.env_id, seed=c.seed, quiet=True)
    if kind == "dqn":
        a = DQNAgent(e.observation_space.shape[0], e.action_space.n, c)
        r, _ = train_dqn(a, e, c, num_episodes=CAUTIOUS_EPISODES, verbose=False)
    else:
        a = SARSAAgent(e.observation_space.shape[0], e.action_space.n, c)
        r, _ = train_sarsa(a, e, c, num_episodes=CAUTIOUS_EPISODES, verbose=False)
    e.close()
    return a, r

dqn_hi,   dqn_hi_r   = run("dqn",   HIGH_EPS)
sarsa_hi, sarsa_hi_r = run("sarsa", HIGH_EPS)

print(f"with the exploration floor raised to {HIGH_EPS}, "
      f"over the last 50 of {CAUTIOUS_EPISODES} episodes:")
print(f"  DQN   {np.mean(dqn_hi_r[-50:]):8.1f}")
print(f"  SARSA {np.mean(sarsa_hi_r[-50:]):8.1f}")

In [ ]:
# how often does the action SARSA bootstraps on coincide with the maximising one?
import torch
for eps in (0.02, HIGH_EPS):
    agree = 1 - eps + eps / n_actions
    print(f"epsilon = {eps:.2f}: the epsilon-greedy action is the maximiser "
          f"{agree:.1%} of the time")
print("\n-> at the end of the main run the two targets coincide on "
      "roughly 98% of steps, which is why the curves above are so similar")

Treat the numbers from the raised-$\epsilon$ run as indicative rather
than conclusive: one seed over a couple of hundred episodes on a noisy
environment is not evidence of much. The argument for the difference is
structural, and the experiment is there to make it concrete, not to prove it.

The honest summary of the whole comparison is the one the deck gives: on this
problem the two perform similarly, and the interesting content is *why* they
should -- a decaying $\epsilon$ drives an on-policy method towards the
off-policy one, by construction.